In [1]:
from dotenv import load_dotenv
import os

load_dotenv(r"F:\Advance RAG\.env", override=True)

WEAVIATE_URL = os.getenv("WEAVIATE_CLUSTER")
WEAVIATE_API_KEY = os.getenv("WEAVIATE_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")

In [2]:
print(repr(WEAVIATE_URL))

'https://weotj5k5q1mzslz0pq0m9w.c0.eu-central-1.aws.weaviate.cloud'


In [4]:
import weaviate

client = weaviate.Client(
    url=WEAVIATE_URL,
    auth_client_secret=weaviate.AuthApiKey(WEAVIATE_API_KEY),
    additional_headers={
        "X-HuggingFace-Api-Key": HF_TOKEN
    }
)

In [38]:
client.is_ready()

True

In [39]:
client.schema.delete_class("RAG")

In [40]:
schema = {
    "classes": [
        {
            "class": "RAG",
            "description": "Documents for RAG",
            "vectorizer": "text2vec-huggingface",
            "moduleConfig": {"text2vec-huggingface": {"model": "sentence-transformers/all-MiniLM-L6-v2", "type": "text"}},
            "properties": [
                {
                    "dataType": ["text"],
                    "description": "The content of the paragraph",
                    "moduleConfig": {
                        "text2vec-huggingface": {
                            "skip": False,
                            "vectorizePropertyName": False,
                        }
                    },
                    "name": "content",
                },
            ],
        },
    ]
}


In [41]:
client.schema.create(schema)

In [42]:

client.schema.get()

{'classes': [{'class': 'RAG',
   'description': 'Documents for RAG',
   'invertedIndexConfig': {'bm25': {'b': 0.75, 'k1': 1.2},
    'cleanupIntervalSeconds': 60,
    'stopwords': {'additions': None, 'preset': 'en', 'removals': None},
    'usingBlockMaxWAND': True},
   'moduleConfig': {'text2vec-huggingface': {'model': 'sentence-transformers/all-MiniLM-L6-v2',
     'type': 'text',
     'useCache': True,
     'useGPU': False,
     'vectorizeClassName': True,
     'waitForModel': False}},
   'multiTenancyConfig': {'autoTenantActivation': False,
    'autoTenantCreation': False,
    'enabled': False},
   'properties': [{'dataType': ['text'],
     'description': 'The content of the paragraph',
     'indexFilterable': True,
     'indexRangeFilters': False,
     'indexSearchable': True,
     'moduleConfig': {'text2vec-huggingface': {'skip': False,
       'vectorizePropertyName': False}},
     'name': 'content',
     'tokenization': 'word'}],
   'shardingConfig': {'actualCount': 1,
    'actualV

In [8]:
from langchain_community.retrievers import WeaviateHybridSearchRetriever

In [66]:
retriever = WeaviateHybridSearchRetriever(
    client=client,
    index_name="RAG",
    text_key="content",
    alpha=0.5,
    k=4,
    attributes=[],
    create_schema_if_missing=False,
)

In [10]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline
)

from langchain_community.llms import HuggingFacePipeline

In [11]:
def load_model(model_name: str):

    model = AutoModelForCausalLM.from_pretrained(
        model_name
    )

    return model

In [12]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [13]:
model = load_model(model_name)

In [14]:
# initializing tokenizer
def initialize_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        return_token_type_ids=False
    )

    tokenizer.bos_token_id = 1

    return tokenizer

In [15]:
tokenizer = initialize_tokenizer(model_name)

In [16]:
from transformers import pipeline

text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="cpu",
    do_sample=False,
    max_new_tokens=20,
    use_cache=False,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [17]:
llm = HuggingFacePipeline(pipeline=text_pipeline)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_3252\326805344.py:1: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=text_pipeline)


In [18]:
doc_path = r"../Data/Retrieval-Augmented-Generation-for-NLP.pdf"

In [19]:
from langchain_community.document_loaders import PyPDFLoader

In [20]:
loader = PyPDFLoader(doc_path)

In [22]:
docs = loader.load()

In [23]:
docs

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Shangyu Wu; Ying Xiong; Yufei Cui; Haolun Wu; Can Chen; Ye Yuan; Lianming Huang; Xue Liu; Tei-Wei Kuo; Nan Guan; Chun Jason Xue', 'doi': 'https://doi.org/10.48550/arXiv.2407.13193', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Retrieval-Augmented Generation for Natural Language Processing: A Survey', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2407.13193v4', 'source': '../Data/Retrieval-Augmented-Generation-for-NLP.pdf', 'total_pages': 67, 'page': 0, 'page_label': '1'}, page_content='Retrieval-Augmented Generation for Natural\nLanguage Processing: A Survey\nShangyu Wu1,2, Ying Xiong2†, Yufei Cui3, Haolun Wu3, Can\nChen3, Ye Yuan3, Lianming Huang1, Xue Liu2, Tei-Wei Kuo4,\nNan Guan1, Chun Jason Xue2\n1City University of Ho

In [25]:
docs[0]

Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Shangyu Wu; Ying Xiong; Yufei Cui; Haolun Wu; Can Chen; Ye Yuan; Lianming Huang; Xue Liu; Tei-Wei Kuo; Nan Guan; Chun Jason Xue', 'doi': 'https://doi.org/10.48550/arXiv.2407.13193', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Retrieval-Augmented Generation for Natural Language Processing: A Survey', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2407.13193v4', 'source': '../Data/Retrieval-Augmented-Generation-for-NLP.pdf', 'total_pages': 67, 'page': 0, 'page_label': '1'}, page_content='Retrieval-Augmented Generation for Natural\nLanguage Processing: A Survey\nShangyu Wu1,2, Ying Xiong2†, Yufei Cui3, Haolun Wu3, Can\nChen3, Ye Yuan3, Lianming Huang1, Xue Liu2, Tei-Wei Kuo4,\nNan Guan1, Chun Jason Xue2\n1City University of Hon

In [26]:
docs[0].metadata

{'producer': 'pikepdf 8.15.1',
 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)',
 'creationdate': '',
 'author': 'Shangyu Wu; Ying Xiong; Yufei Cui; Haolun Wu; Can Chen; Ye Yuan; Lianming Huang; Xue Liu; Tei-Wei Kuo; Nan Guan; Chun Jason Xue',
 'doi': 'https://doi.org/10.48550/arXiv.2407.13193',
 'license': 'http://creativecommons.org/licenses/by/4.0/',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1',
 'title': 'Retrieval-Augmented Generation for Natural Language Processing: A Survey',
 'trapped': '/False',
 'arxivid': 'https://arxiv.org/abs/2407.13193v4',
 'source': '../Data/Retrieval-Augmented-Generation-for-NLP.pdf',
 'total_pages': 67,
 'page': 0,
 'page_label': '1'}

In [44]:
len(docs)

67

In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [29]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

split_docs = text_splitter.split_documents(docs)

In [45]:
print(len(split_docs))

441


In [46]:
for doc in split_docs:
    doc.metadata = {
        k.replace(".", "_").replace("-", "_").replace(" ", "_"): v
        for k, v in doc.metadata.items()
    }

In [47]:
retriever.add_documents(split_docs)

['9d351666-49d5-40ce-91f3-1507f1115fad',
 'cf14d8f0-c11a-4830-9fb1-3ea11d2a625a',
 'a255bdb8-b7b4-4a89-a83e-9454a04604f8',
 '22ef727f-4319-4f13-b37a-c1d619ef4b63',
 'c9870a0f-4ad2-4cce-972c-ff81bd4d9487',
 'dae8da71-de31-4df9-9e8a-8ab3c4e78b51',
 '36f6098e-df37-4a50-b063-41181ff9b5fa',
 'f3e33c98-cb37-4cbd-89c9-cf0398eac33c',
 'e9c11b63-858c-4802-be0b-3a3f727f64f9',
 '13b53be5-f5b1-4074-a7bf-07dccec4fd6b',
 '5eab6bb0-81f2-4441-8228-a620feb2102e',
 '22a7de73-cd15-4794-9d0d-bec73f4731f9',
 '132f3c3b-46d7-4377-a2b2-b6a04410b147',
 '2226c9c0-ccf1-4598-a829-33176907dd23',
 'ba1b9bf5-a840-41a6-8fee-8a10f9ddca00',
 'eb3fccf8-40fb-4e43-b5d9-5796a51f9691',
 '8b4a6e70-d448-4a3a-aba0-b42856c1a89f',
 '977b8c0b-9b39-4ac7-bb27-78fb2baadde1',
 '099d3e64-4d8c-46de-9959-d01f82614aac',
 'cf1774a0-8d3b-4f78-9e5a-208c5b5df1af',
 '3dd3ee36-4d77-4ae4-9bcd-71bb42ba3add',
 '5f507f87-01ec-43bc-967a-2d974cf7d79e',
 '685a4a43-4cfb-420a-aa45-250d73b22639',
 '40346cc9-eb8e-4867-a0e9-779cee153ffc',
 '5271cba2-14bc-

In [49]:
results = retriever.invoke("What is Retrieval-Augmented Generation?")

In [50]:
len(results)

2

In [51]:
print(results[0].page_content)

cient retrieval and data persistence. A key design decision in the knowledge base is
what should be stored as values. For example, for question-answer tasks, when adding
retrievals to prompts, the naive but effective way is to store the question embedding
as the key and question-answer pairs as the value. This can help the generation pro-
cess as retrievals are used as demonstrations for models. Recent works have proposed


In [52]:
retriever.invoke(
    "what is RAG token?",
    score=True
)

[Document(metadata={'_additional': {'explainScore': '\nHybrid (Result Set keyword,bm25) Document 3526b5c9-780c-49fd-ba23-4362cf7aa8d8: original score 2.4784236, normalized score: 0.47461995 - \nHybrid (Result Set vector,hybridVector) Document 3526b5c9-780c-49fd-ba23-4362cf7aa8d8: original score 0.5199056, normalized score: 0.5', 'score': '0.97462'}}, page_content='these tools represent a shift from bespoke RAG implementations to standardized,\nreusable, and often language-agnostic development paradigms.\n9.3 Deployment Considerations for Production RAG\nAlthough the RAG pattern is conceptually simple, production systems face multi-\ndimensional constraints, including retrieval quality, scalability, token/context budgets,\nlatency, security/governance, and evaluation/observability (Microsoft 2026b). In this'),
 Document(metadata={'_additional': {'explainScore': '\nHybrid (Result Set keyword,bm25) Document 62fec4e4-a62e-4fa6-a995-090fe0ef5d7e: original score 2.434456, normalized score: 0

In [53]:
from langchain.chains import RetrievalQA

In [93]:

from langchain_core.prompts import ChatPromptTemplate

In [94]:

system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)
     

In [95]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{query}"),
    ]
)

In [96]:

from langchain.prompts import PromptTemplate
template = """
Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you do not have the relevant information needed to provide a verified answer, don't try to make up an answer.
When providing an answer, aim for clarity and precision. Position yourself as a knowledgeable authority on the topic, but also be mindful to explain the information in a manner that is accessible and comprehensible to those without a technical background.
Always say "Do you have any more questions pertaining to this instrument?" at the end of the answer.
{context}
Question: {question}
Helpful Answer:"""

prompt = PromptTemplate.from_template(template)
     

In [97]:

from langchain.chains.combine_documents import create_stuff_documents_chain

In [98]:

question_answer_chain = create_stuff_documents_chain(llm, prompt)

In [71]:
hybrid_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)


In [72]:
result1 = hybrid_chain.invoke("what is RAG Token?")
print(result1)

{'query': 'what is RAG Token?', 'result': "Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\nthese tools represent a shift from bespoke RAG implementations to standardized,\nreusable, and often language-agnostic development paradigms.\n9.3 Deployment Considerations for Production RAG\nAlthough the RAG pattern is conceptually simple, production systems face multi-\ndimensional constraints, including retrieval quality, scalability, token/context budgets,\nlatency, security/governance, and evaluation/observability (Microsoft 2026b). In this\n\nRajpurkar P, Jia R, Liang P (2018) Know what you don’t know: Unanswerable ques-\ntions for squad. In: Proceedings of the 56th Annual Meeting of the Association for\nComputational Linguistics (ACL), pp 784–789\nRam O, Bezalel L, Zicher A, et al (2023a) What are you token about? dense retrieval\nas distributions over the vocabulary. In: 

In [73]:
print(result1['result'])

Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

these tools represent a shift from bespoke RAG implementations to standardized,
reusable, and often language-agnostic development paradigms.
9.3 Deployment Considerations for Production RAG
Although the RAG pattern is conceptually simple, production systems face multi-
dimensional constraints, including retrieval quality, scalability, token/context budgets,
latency, security/governance, and evaluation/observability (Microsoft 2026b). In this

Rajpurkar P, Jia R, Liang P (2018) Know what you don’t know: Unanswerable ques-
tions for squad. In: Proceedings of the 56th Annual Meeting of the Association for
Computational Linguistics (ACL), pp 784–789
Ram O, Bezalel L, Zicher A, et al (2023a) What are you token about? dense retrieval
as distributions over the vocabulary. In: Proceedings of the 61st Annual Meeting of
the Associatio

In [78]:
result1["query"]

'what is RAG Token?'

In [80]:
from pprint import pprint

pprint(result1)

{'query': 'what is RAG Token?',
 'result': 'Use the following pieces of context to answer the question at the '
           "end. If you don't know the answer, just say that you don't know, "
           "don't try to make up an answer.\n"
           '\n'
           'these tools represent a shift from bespoke RAG implementations to '
           'standardized,\n'
           'reusable, and often language-agnostic development paradigms.\n'
           '9.3 Deployment Considerations for Production RAG\n'
           'Although the RAG pattern is conceptually simple, production '
           'systems face multi-\n'
           'dimensional constraints, including retrieval quality, scalability, '
           'token/context budgets,\n'
           'latency, security/governance, and evaluation/observability '
           '(Microsoft 2026b). In this\n'
           '\n'
           'Rajpurkar P, Jia R, Liang P (2018) Know what you don’t know: '
           'Unanswerable ques-\n'
           'tions for squad. 

In [99]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

In [100]:
# Set up the RAG chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()} |
    prompt |
    llm
)

In [101]:
query="what is RAG token?"

In [102]:
response=rag_chain.invoke("what is RAG token?")

In [103]:
print(response)


Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you do not have the relevant information needed to provide a verified answer, don't try to make up an answer.
When providing an answer, aim for clarity and precision. Position yourself as a knowledgeable authority on the topic, but also be mindful to explain the information in a manner that is accessible and comprehensible to those without a technical background.
Always say "Do you have any more questions pertaining to this instrument?" at the end of the answer.
[Document(metadata={}, page_content='these tools represent a shift from bespoke RAG implementations to standardized,\nreusable, and often language-agnostic development paradigms.\n9.3 Deployment Considerations for Production RAG\nAlthough the RAG pattern is conceptually simple, production systems face multi-\ndimensional constraints, including retrieval quality, scalability, token/context budgets,\nlatency, securi

In [105]:
from pprint import pprint

pprint(response)

('\n'
 'Use the following pieces of context to answer the question at the end.\n'
 "If you don't know the answer, just say that you do not have the relevant "
 "information needed to provide a verified answer, don't try to make up an "
 'answer.\n'
 'When providing an answer, aim for clarity and precision. Position yourself '
 'as a knowledgeable authority on the topic, but also be mindful to explain '
 'the information in a manner that is accessible and comprehensible to those '
 'without a technical background.\n'
 'Always say "Do you have any more questions pertaining to this instrument?" '
 'at the end of the answer.\n'
 "[Document(metadata={}, page_content='these tools represent a shift from "
 'bespoke RAG implementations to standardized,\\nreusable, and often '
 'language-agnostic development paradigms.\\n9.3 Deployment Considerations for '
 'Production RAG\\nAlthough the RAG pattern is conceptually simple, production '
 'systems face multi-\\ndimensional constraints, including 

In [56]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank

In [81]:
load_dotenv()

compressor = CohereRerank(
    cohere_api_key=os.getenv("cohere_api_key"),
    model="rerank-v3.5",
    top_n=3
)

In [82]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

In [83]:
results = compression_retriever.invoke(
    "What is RAG Token?"
)

print(results)

[Document(metadata={'relevance_score': 0.2699866}, page_content='these tools represent a shift from bespoke RAG implementations to standardized,\nreusable, and often language-agnostic development paradigms.\n9.3 Deployment Considerations for Production RAG\nAlthough the RAG pattern is conceptually simple, production systems face multi-\ndimensional constraints, including retrieval quality, scalability, token/context budgets,\nlatency, security/governance, and evaluation/observability (Microsoft 2026b). In this'), Document(metadata={'relevance_score': 0.22893171}, page_content='steps (Databricks 2025b). Similarly, Azure distinguishes classic RAG and agentic\nretrieval for complex queries, reflecting a practical trade-off between quality and\nlatency (Microsoft 2026b,a).\nCost-Performance Trade-Offs.Key cost drivers include ingestion/embedding,\nvector search, reranking, and LLM token usage. Vertex AI RAG Engine explicitly item-\nizes billing across ingestion/parsing, embeddings, vector 

In [84]:
from pprint import pprint

pprint(results)

[Document(metadata={'relevance_score': 0.2699866}, page_content='these tools represent a shift from bespoke RAG implementations to standardized,\nreusable, and often language-agnostic development paradigms.\n9.3 Deployment Considerations for Production RAG\nAlthough the RAG pattern is conceptually simple, production systems face multi-\ndimensional constraints, including retrieval quality, scalability, token/context budgets,\nlatency, security/governance, and evaluation/observability (Microsoft 2026b). In this'),
 Document(metadata={'relevance_score': 0.22893171}, page_content='steps (Databricks 2025b). Similarly, Azure distinguishes classic RAG and agentic\nretrieval for complex queries, reflecting a practical trade-off between quality and\nlatency (Microsoft 2026b,a).\nCost-Performance Trade-Offs.Key cost drivers include ingestion/embedding,\nvector search, reranking, and LLM token usage. Vertex AI RAG Engine explicitly item-\nizes billing across ingestion/parsing, embeddings, vector

In [ ]:
compressed_docs = compression_retriever.get_relevant_documents(user_query)
# Print the relevant documents from using the embeddings and reranker
print(compressed_docs)

In [86]:
hybrid_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=compression_retriever
)

In [87]:
response = hybrid_chain.invoke("What is RAG Token?")

f:\Advance RAG\rag_env\Lib\site-packages\packaging\version.py:207: ResourceWarning: unclosed <ssl.SSLSocket fd=1496, family=2, type=1, proto=0, laddr=('192.168.18.2', 51849), raddr=('34.96.76.122', 443)>
  release=tuple(int(i) for i in match.group("release").split(".")),
f:\Advance RAG\rag_env\Lib\site-packages\packaging\version.py:207: ResourceWarning: unclosed <ssl.SSLSocket fd=1208, family=2, type=1, proto=0, laddr=('192.168.18.2', 51886), raddr=('34.96.76.122', 443)>
  release=tuple(int(i) for i in match.group("release").split(".")),


In [88]:
print(response["result"]   )

Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

these tools represent a shift from bespoke RAG implementations to standardized,
reusable, and often language-agnostic development paradigms.
9.3 Deployment Considerations for Production RAG
Although the RAG pattern is conceptually simple, production systems face multi-
dimensional constraints, including retrieval quality, scalability, token/context budgets,
latency, security/governance, and evaluation/observability (Microsoft 2026b). In this

steps (Databricks 2025b). Similarly, Azure distinguishes classic RAG and agentic
retrieval for complex queries, reflecting a practical trade-off between quality and
latency (Microsoft 2026b,a).
Cost-Performance Trade-Offs.Key cost drivers include ingestion/embedding,
vector search, reranking, and LLM token usage. Vertex AI RAG Engine explicitly item-
izes billing across ingestion/parsin

In [90]:
from pprint import pprint

pprint(response)

{'query': 'What is RAG Token?',
 'result': 'Use the following pieces of context to answer the question at the '
           "end. If you don't know the answer, just say that you don't know, "
           "don't try to make up an answer.\n"
           '\n'
           'these tools represent a shift from bespoke RAG implementations to '
           'standardized,\n'
           'reusable, and often language-agnostic development paradigms.\n'
           '9.3 Deployment Considerations for Production RAG\n'
           'Although the RAG pattern is conceptually simple, production '
           'systems face multi-\n'
           'dimensional constraints, including retrieval quality, scalability, '
           'token/context budgets,\n'
           'latency, security/governance, and evaluation/observability '
           '(Microsoft 2026b). In this\n'
           '\n'
           'steps (Databricks 2025b). Similarly, Azure distinguishes classic '
           'RAG and agentic\n'
           'retrieval for co